# Population snapshot 2014 — normalisation & anchor validation

**Pipeline notebook 4 (Phase 3 input).** `BDCH2014.csv` is a year-end 2014 *stock*
snapshot: one row per resident present on 2014-12-31, on the same `NOREFCH` key as
the mutations (`Annee = 2014`, `Norefch = NOREFCH`, confirmed). It follows a third
schema — neither the old semi-annual nor the new monthly one — so an explicit rename
map is required. Dates are Swiss `DD.MM.YYYY` (not Excel serial).

The snapshot is kept as a separate `population_2014` table (state grain, not event
grain). It serves two roles downstream: **anchor** (seed episode 0 of pre-2015
residents with their real 2014 state) and **validation** (compare reconstruction to
the true year-end stock). `Idfam`/`Rolefam` are retained: they carry a ready-made
family nucleus — absent from the monthly schema — used later to *validate* (not
build) the inferred household typology.


## 0. Load the snapshot and the audited master

The snapshot is read with the robust reader (encoding fallback for accents,
separator detection). The audited master is loaded for the coherence checks.


In [6]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/mnt/o/09_STATISTIQUES/09_04_Explorations/MR/CAS ADS/Final project/data")
SNAP     = DATA_DIR / "BDCH2014.csv"

def _detect_sep(path, enc):
    with open(path, encoding=enc, newline="") as f:
        first = f.readline()
    return max([",", ";", "\t"], key=first.count)

def read_robust(path):
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            sep = _detect_sep(path, enc)
            df  = pd.read_csv(path, sep=sep, encoding=enc, dtype=str, low_memory=False)
            df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
            return df, enc, sep
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"decode failed: {path}")

snap, enc, sep = read_robust(SNAP)
print(f"BDCH2014: {len(snap):,} rows x {snap.shape[1]} cols | enc={enc} sep={sep!r}")

# Audited master for validation
for f in [DATA_DIR/"masterfile_events.parquet", DATA_DIR/"masterfile_events.csv.gz",
          DATA_DIR/"masterfile_audited.parquet", DATA_DIR/"masterfile_audited.csv.gz"]:
    if f.exists():
        clean = pd.read_parquet(f) if f.suffix == ".parquet" else pd.read_csv(f, dtype=str, low_memory=False)
        print("Master loaded:", f.name, "->", f"{len(clean):,} rows")
        break
else:
    raise FileNotFoundError("Run notebooks 2/3 first.")

for c in ["DATE_EFFECTIVE", "MUTATION_DATE"]:
    if c in clean.columns and not pd.api.types.is_datetime64_any_dtype(clean[c]):
        clean[c] = pd.to_datetime(clean[c], format="%d.%m.%Y %H:%M", errors="coerce")

BDCH2014: 140,228 rows x 58 cols | enc=cp1252 sep=';'
Master loaded: masterfile_events.parquet -> 857,069 rows


## 1. Normalise to the target schema

Columns are renamed to the monthly schema so the snapshot joins on `id_projet`.
Dates are parsed (Swiss format) and dwelling numerics coerced. Snapshot-only extras
(`FAMILLE_ID`, `FAMILLE_ROLE`, `DUREE_LS_2014`, quarter/sector, religion) are kept
under clear names. Note `TYPE_MENAGE` is binary here (2014 schema: `Prive`), coarser
than the four-way monthly value — do not compare the labels literally.


In [7]:
RENAME_2014 = {
    "Norefch": "id_projet", "Codetat": "CODETA_C",
    "Datnais": "DATNAIS", "Sexe": "CODE_SEXE", "Etatcivil": "ETAT_CIVIL",
    "NationCo": "NATION", "PaysnaisCo": "PAYSNAIS", "ComnaisCo": "COMNAIS",
    "DatarrLS": "DATARR", "Datdem": "DATDEM", "DatentCH": "DATEENTCH",
    "PaysprovCo": "PAYSPROVN", "ComprovCo": "COMPROV",
    "Egid": "EGID", "Ewid": "MENAGE_EWID", "NocomOFS": "NOCOM",
    "Idmen": "NUMERO_MENAGE", "Typmen": "TYPE_MENAGE",
    "Pièces": "NOMBRE_PIECE", "Surface": "SURFACE",
    "Idfam": "FAMILLE_ID", "Rolefam": "FAMILLE_ROLE", "DureeLS": "DUREE_LS_2014",
    "Quartier": "QUARTIER", "Secteur": "SECTEUR", "Religion": "RELIGION",
}

pop2014 = snap.rename(columns=RENAME_2014)
pop2014["snapshot_date"] = pd.Timestamp("2014-12-31")

for c in ["DATNAIS", "DATARR", "DATDEM", "DATEENTCH"]:
    if c in pop2014.columns:
        pop2014[c] = pd.to_datetime(pop2014[c], format="%d.%m.%Y", errors="coerce")
for c in ["NOMBRE_PIECE", "SURFACE"]:
    if c in pop2014.columns:
        pop2014[c] = pd.to_numeric(pop2014[c].astype(str).str.replace(",", ".", regex=False),
                                   errors="coerce")

print(f"population_2014: {len(pop2014):,} residents | "
      f"unique id_projet: {pop2014['id_projet'].nunique():,} | "
      f"dup id: {pop2014.duplicated('id_projet').sum():,}")

population_2014: 140,228 residents | unique id_projet: 140,228 | dup id: 0


## 2. Validation 1 — overlap with the mutation corpus

How many 2014 residents reappear in the 2015+ mutations. Those never seen again are
the "dormant" stock the snapshot uniquely reveals: present at year-end 2014, then no
captured mutation (stable then left/died without an event, or still present without a
mutation). This is precisely the left-truncation gap the snapshot fills.


In [8]:
ids_mut  = set(clean["id_projet"].dropna())
ids_2014 = set(pop2014["id_projet"].dropna())
inter    = ids_2014 & ids_mut

print(f"2014 residents also seen in mutations: {len(inter):,} "
      f"({len(inter)/len(ids_2014)*100:.1f}% of 2014)")
print(f"2014 residents never seen after 2015 : {len(ids_2014 - ids_mut):,}")
print(f"Individuals in mutations not in 2014 : {len(ids_mut - ids_2014):,} "
      f"(arrived / born 2015+)")

2014 residents also seen in mutations: 99,661 (71.1% of 2014)
2014 residents never seen after 2015 : 40,567
Individuals in mutations not in 2014 : 191,266 (arrived / born 2015+)


## 3. Validation 2 — 2014 state vs first post-2015 before-state

The strong check: for residents who did not move between year-end 2014 and their
first post-2015 mutation, the 2014 state should equal that mutation's `MUTATION_*`
before-state. High agreement confirms both the before/after column model and the key
join. Lower agreement is largely explained by moves in the interval (filterable via
`DATDEM` > 2014), which should be inspected rather than assumed away.


In [9]:
first_mut = (clean[clean["id_projet"].isin(inter)]
             .sort_values(["id_projet", "DATE_EFFECTIVE", "MUTATION_DATE"])
             .groupby("id_projet").head(1)
             .set_index("id_projet"))
p14 = pop2014.set_index("id_projet")

checks = {"EGID": "MUTATION_EGID", "MENAGE_EWID": "MUTATION_EWID",
          "NUMERO_MENAGE": "MUTATION_NUMERO_MENAGE"}
print("State agreement (2014 value == before-state of first post-2015 mutation):")
for snap_col, prev_col in checks.items():
    if snap_col in p14.columns and prev_col in first_mut.columns:
        a = p14[snap_col].reindex(first_mut.index)
        b = first_mut[prev_col]
        m = a.notna() & b.notna()
        agree = (a[m].astype(str).str.strip() == b[m].astype(str).str.strip()).mean() if m.any() else float("nan")
        print(f"  {snap_col:14} vs {prev_col:24}: {agree*100:5.1f}% agree / {int(m.sum()):,} comparable")

State agreement (2014 value == before-state of first post-2015 mutation):
  EGID           vs MUTATION_EGID           :  95.7% agree / 99,239 comparable
  MENAGE_EWID    vs MUTATION_EWID           :  93.3% agree / 94,865 comparable
  NUMERO_MENAGE  vs MUTATION_NUMERO_MENAGE  :  95.5% agree / 99,372 comparable


## 4. Save

`population_2014` is saved as the anchor/validation table for Phase 3. It is **not**
merged into the event master (different grain: state vs event).


In [10]:
pq = DATA_DIR / "population_2014.parquet"
try:
    pop2014.to_parquet(pq, index=False)
    saved = pq
except Exception as e:
    saved = pq.with_suffix(".csv.gz")
    pop2014.to_csv(saved, index=False, compression="gzip")
    print(f"Parquet unavailable ({type(e).__name__}) -> CSV gzip.")
print(f"Saved: {saved} ({saved.stat().st_size/1e6:.1f} MB)")

Saved: /mnt/o/09_STATISTIQUES/09_04_Explorations/MR/CAS ADS/Final project/data/population_2014.parquet (9.7 MB)
